---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

---

**Buat Spark Session terlebih dahulu**

Jalankan kode pada cell dibawah ini:

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, count, sum as spark_sum, avg

spark = SparkSession.builder \
    .appName("Tugas6_ETL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark siap:", spark.version)

26/09/24 11:20:40 WARN Utils: Your hostname, angger resolves to a loopback address: 127.0.1.1; using 192.168.1.163 instead (on interface wlp3s0)
26/09/24 11:20:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 11:20:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark siap: 3.5.9


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [6]:
# Membaca file
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header = True,
    inferSchema = True)

df_produk = spark.read.json(
    "tugas6_produk.json")

df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header = True,
    inferSchema = True)

# Menampilkan masing-maisng jumlah

print("Jumlah Transaksi: ", df_transaksi.count())
print("Jumlah Produk: ", df_produk.count())
print("Jumlah Ulasan: ", df_ulasan.count())

# Menampilkan scheme

print("=== SCHEMA TRANSAKSI ===")
df_transaksi.printSchema()

print("=== SCHEMA PRODUK ===")
df_produk.printSchema()

print("=== SCHEMA ULASAN ===")
df_ulasan.printSchema()

Jumlah Transaksi:  5000
Jumlah Produk:  30
Jumlah Ulasan:  3500
=== SCHEMA TRANSAKSI ===
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

=== SCHEMA PRODUK ===
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

=== SCHEMA ULASAN ===
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).


In [18]:
# Menggabungkan tabel transaksi dengan tabel produk
df_trns_produk = df_transaksi.join(
    df_produk,
    on = "product_id",
    how = "inner")

# Menggabungkan hasil dengan ulasan
df_etl = df_trns_produk.join(
    df_ulasan,
    on = "order_id",
    how = "left")

# Menghitung total pendapatan
df_etl = df_etl.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga"))

# Tampilkan hasilnnya
df_etl.select(
    "order_id",
    "product_id",
    "nama_produk",
    "kategori",
    "unit_terjual",
    "harga",
    "rating",
    "total_pendapatan"
).show(10)

+--------+----------+-----------+------------+------------+------+------+----------------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|rating|total_pendapatan|
+--------+----------+-----------+------------+------------+------+------+----------------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|     1|           50000|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|  NULL|         1500000|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|     2|          100000|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|     5|          300000|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|  NULL|          450000|
|     TX5|        10|  Produk-10|  Elektronik|           3| 75000|     5|          225000|
|     TX6|        26|  Produk-26|   Kesehatan|           6| 75000|     4|          450000|
|     TX7|         4|   Produk-4|     Makanan|           1|150000|  NULL|          150000|

In [19]:
print("Jumlah baris setelah join:", df_etl.count())

Jumlah baris setelah join: 5000


**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

In [20]:
# Nambahin kolom yang ada ulasannya
df_etl = df_etl.withColumn(
    "ada_ulasan",
    col("rating").isNotNull())

# Isi colom ulasan yang null pakai 0
df_etl = df_etl.na.fill({
    "rating":0})

# Seleksi hal yang berbau rating
df_etl.select(
    "order_id",
    "rating",
    "ada_ulasan").show(8)

# Cel jumlah transaksi
df_etl.groupBy("ada_ulasan").count().show()


+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|     0|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|     0|     false|
|     TX5|     5|      true|
|     TX6|     4|      true|
|     TX7|     0|     false|
+--------+------+----------+
only showing top 8 rows

+----------+-----+
|ada_ulasan|count|
+----------+-----+
|      true| 3500|
|     false| 1500|
+----------+-----+



**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

In [21]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

df_etl.write \
.mode("overwrite") \
.partitionBy("kategori") \
.parquet(output_path)

print("CData berhasil disimpan ke HDFS")

[Stage 67:>                                                         (0 + 1) / 1]

CData berhasil disimpan ke HDFS


**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?


In [22]:
from pyspark.sql.functions import count, sum as spark_sum, when, col, round

insight = df_etl.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    spark_sum(
        when(col("ada_ulasan"), 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
).withColumn(
    "persentase_dengan_ulasan",
    round(
        col("transaksi_dengan_ulasan") /
        col("total_transaksi") * 100,
        2
    )
).orderBy("persentase_dengan_ulasan")

insight.show()
kategori_terendah = insight.first()

print("Kategori dengan persentase ulasan terendah:",
      kategori_terendah["kategori"])

print("Persentase transaksi dengan ulasan:",
      kategori_terendah["persentase_dengan_ulasan"], "%")

+------------+---------------+-----------------------+------------------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_dengan_ulasan|
+------------+---------------+-----------------------+------------------------+
|     Makanan|            536|                    369|                   68.84|
|   Kesehatan|            961|                    662|                   68.89|
|     Fashion|           1035|                    726|                   70.14|
|Rumah Tangga|            815|                    575|                   70.55|
|  Elektronik|           1653|                   1168|                   70.66|
+------------+---------------+-----------------------+------------------------+

Kategori dengan persentase ulasan terendah: Makanan
Persentase transaksi dengan ulasan: 68.84 %


Dari data hasil hasil ETL, kategori makanan ini punya persentase transaksi dengan ulasan paling rendah, yaitu 68,84%. Nahh ini menunjukkan bahwa transaksi pada kategori Makanan memiliki tingkat pemberian ulasan yang relatif lebih rendah dibandingkan kategori lainnya. Informasi ini dapat menjadi perhatian bagi tim marketing untuk meningkatkan engagement pelanggan, misalnya dengan memberikan pengingat atau ajakan untuk memberikan ulasan setelah melakukan pembelian.

---